## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gemini-3.5-flash-lite"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatGoogleGenerativeAI(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='63a99255-ed0a-4d36-b7f1-05822cc15d4c', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
llm.invoke("Who is Avery?")

c:\Users\NimashaTennakoon\OneDrive - Arcadea Group\Desktop\Personal Development\ai-engineer-core-track-udemy\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[{'type': 'text', 'text': 'Because "Avery" is a very common first and last name, and can refer to several different people, fictional characters, or things, **could you please specify which Avery you are asking about?** \n\nHere are a few famous or notable examples of people and characters named Avery:\n\n### Historical & Public Figures\n* **Avery Brundage:** An American sports official who served as the fifth President of the International Olympic Committee (IOC) from 1952 to 1972.\n* **Tex Avery:** A famous American animator, cartoonist, and director, best known for his work on *Looney Tunes* and *Merrie Melodies* at Warner Bros. and MGM (creating characters like Daffy Duck, Bugs Bunny, and Droopy).\n* **Oswald Avery:** A Canadian-American physician and medical researcher who demonstrated that DNA is the material responsible for heredity (the Avery–MacLeod–McCarty experiment).\n\n### Fictional Characters\n* **Jackson Avery:** A prominent fictional surgeon on the lon

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

c:\Users\NimashaTennakoon\OneDrive - Arcadea Group\Desktop\Personal Development\ai-engineer-core-track-udemy\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text',
  'text': 'Avery Lancaster (often spelled Averi) is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been leading the company since she co-founded it in 2015, helping guide Insurellm to become a leading Insurance Tech provider through her innovative leadership strategies and risk management expertise. \n\nBefore starting Insurellm, Avery worked as a Senior Product Manager at Innovate Insurance Solutions from 2013 to 2015, where she developed groundbreaking insurance products for the tech sector. She is based in San Francisco, California.',
  'extras': {'signature': 'EmAKXgFpFH0TvYr4vwYhO2y4n/nYb2MOR920ZRFZnw6qiqcOlZkNw3XTatrG9hKqw2gm28t5OCSymkseA34zQAdcqHkg+bUEzZZc+yTQQprtQGi4RG3fL0leAt9ILY/GsqU='}}]

## What could possibly come next? 😂

In [10]:
gr.ChatInterface(answer_question).launch()

c:\Users\NimashaTennakoon\OneDrive - Arcadea Group\Desktop\Personal Development\ai-engineer-core-track-udemy\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\NimashaTennakoon\OneDrive - Arcadea Group\Desktop\Personal Development\ai-engineer-core-track-udemy\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\NimashaTennakoon\OneDrive - Arcadea Group\Desktop\Personal Development\ai-engineer-core-track-udemy\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\NimashaTennakoon\OneDrive - Arcadea Group\Desktop\Personal Development\ai-engineer-core-track-udemy\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  req

## Admit it - you thought RAG would be more complicated than that!!